In [ ]:
# Placeholder parameters: illustrative values only, not the ones used in live research.
THRESHOLDS = [0.5, 1.0, 1.5, 2.0]  # 個人智慧財產,參數不提供 (placeholder)
PROFITS = [10, 8, 5, 3]  # 個人智慧財產,參數不提供 (placeholder)
LOOKBACK_DAYS = 5  # 個人智慧財產,參數不提供 (placeholder)
VOL_CUTOFF = 20  # 個人智慧財產,參數不提供 (placeholder)

# Assumptions
# IV effect on option price/IV effect on risk = c
Execute when VIX < X

sell call strike賣比current price高 X%

X個禮拜內SPY跌幅高於X%就不執行

# X天內跌超過threshold就不執行

In [ ]:
import pandas as pd


# Load and prepare data
# df = pd.read_csv("QQQ ETF Stock Price History.csv")
df = pd.read_csv("SPY ETF Stock Price History 2021 to 2025.csv")
df.rename(columns={'Price': 'Close'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df['Daily % Change'] = df['Close'].pct_change() * 100
df = df.dropna().reset_index(drop=True)
df['Weekday'] = df['Date'].dt.weekday

# Strategy configuration
thresholds = THRESHOLDS
# thresholds = [X, X, X, X, X, X, X, X, X, X]
profits = PROFITS
# profits =    [X, X, X, X, X, X, X, X, X, X]
lookback_days = LOOKBACK_DAYS
thres_mult = 1

daily_moves = df['Daily % Change'].values
weekdays = df['Weekday'].values
results = []

# Backtest strategy
for thres, profit in zip(thresholds, profits):
    loss = 100 - profit
    wins, losses, skips = 0, 0, 0
    num_days = len(daily_moves)
    for i in range(lookback_days, num_days):
        # if any(change < -thres*thres_mult for change in daily_moves[i - lookback_days:i]):
        #     skips += X
        #     continue
        # if daily_moves[i] >= -thres:
        if daily_moves[i] <= thres:
            wins += 1
        else:
            losses += 1

    total_trades = wins + losses
    if total_trades == 0:
        continue

    win_rate = wins / total_trades
    expected_value = (win_rate * profit) - ((1 - win_rate) * loss)
    reward = expected_value * total_trades

    results.append({
        "Breakeven Threshold (%)": thres,
        "Profit if Win (%)": profit,
        "Loss if Breached (%)": loss,
        "Trades Taken": total_trades,
        "Trades Skipped": skips,
        "Chance to Profit (%)": round(win_rate * 100, 2),
        "Chance to Lose (%)": round((1 - win_rate) * 100, 2),
        "Expected Value (%)": round(expected_value, 2),
        "Reward (Profit × Trades)": round(reward, 2)
    })

# Output sorted by reward
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="Reward (Profit × Trades)", ascending=False).reset_index(drop=True)
print(results_df)


# Modify strategy to sell the most ITM contract that fulfills the thresholding

In [ ]:
import pandas as pd

# Load SPY price data
df = pd.read_csv("SPY ETF Stock Price History 2021 to 2025.csv")  # Update with correct path if needed
df.rename(columns={'Price': 'Close'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df['Daily % Change'] = df['Close'].pct_change() * 100
df = df.dropna().reset_index(drop=True)
df['Weekday'] = df['Date'].dt.weekday

# Strategy Parameters
lookback_days = LOOKBACK_DAYS
thres_mult = 1

# Ordered from tightest to loosest thresholds
thresholds = THRESHOLDS
# thresholds = [X, X, X, X, X, X, X, X, X, X]
profits = PROFITS
# profits =    [X,    X,    X,    X,   X,   X,    X,     X,     X,     X]

# Prepare arrays
daily_moves = df['Daily % Change'].values
num_days = len(daily_moves)

# Trade simulation
trade_log = []

for i in range(lookback_days, num_days):
    past_window = daily_moves[i - lookback_days:i]
    max_drop = abs(min(past_window))

    # Find the closest eligible threshold
    selected_index = None
    for j, threshold in enumerate(thresholds):
        if max_drop <= threshold*thres_mult:
            selected_index = j
            break

    if selected_index is None:
        continue  # No eligible contract that fits drop condition

    threshold = thresholds[selected_index]
    profit = profits[selected_index]
    loss = 100 - profit
    result = 'WIN' if daily_moves[i] <= threshold else 'LOSS'

    trade_log.append({
        "Date": df['Date'][i],
        "Threshold (%)": threshold,
        "Profit (%)": profit,
        "Loss (%)": loss,
        "Actual Move (%)": round(daily_moves[i], 2),
        "Result": result
    })

# Convert to DataFrame
trade_df = pd.DataFrame(trade_log)

# Summary Statistics
total_trades = len(trade_df)
wins = (trade_df['Result'] == 'WIN').sum()
losses = total_trades - wins
win_rate = wins / total_trades
expected_value = (win_rate * trade_df['Profit (%)'].mean()) - ((1 - win_rate) * trade_df['Loss (%)'].mean())
reward = expected_value * total_trades

# Print Summary
summary = {
    "Total Trades": total_trades,
    "Wins": wins,
    "Losses": losses,
    "Win Rate (%)": round(win_rate * 100, 2),
    "Average Profit (%)": round(trade_df['Profit (%)'].mean(), 2),
    "Average Loss (%)": round(trade_df['Loss (%)'].mean(), 2),
    "Expected Value per Trade (%)": round(expected_value, 2),
    "Total Reward": round(reward, 2)
}

summary_df = pd.DataFrame([summary])
print(summary_df)

# Optional: Save trade log
# trade_df.to_csv("trade_log.csv", index=False)


# Modify strategy to sell a default contract, when contract is not tradable sell the next tradable contract


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load SPY price data
df = pd.read_csv("SPY ETF Stock Price History 2021 to 2025.csv")
# df = pd.read_csv("QQQ ETF Stock Price History.csv")
df.rename(columns={'Price': 'Close'}, inplace=True)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)
df["Date"] = pd.to_datetime(df["Date"])
df['Daily % Change'] = df['Close'].pct_change() * 100
df['Change %'] = df['Change %'].astype(str).str.replace('%', '').astype(float)
df = df.dropna().reset_index(drop=True)

# Strategy Parameters
lookback_days = LOOKBACK_DAYS
thres_mult = 1

# Thresholds and profits from tightest to loosest
thresholds = THRESHOLDS
# thresholds = [X, X, X, X, X, X, X, X, X, X, X]
profits = PROFITS
# profits =    [X,    X,    X,    X,   X,   X,    X,     X,     X,     X,    X]

# Default contract index (e.g., threshold = X → index X)
default_index = len(thresholds) // 2

daily_moves = df['Daily % Change'].values
num_days = len(daily_moves)

trade_log = []
total_pnl = 0

for i in range(lookback_days, num_days):
    max_drop = abs(min(daily_moves[i - lookback_days:i]))

    # Start from default and go to looser contracts
    selected_index = None
    if max_drop <= thresholds[default_index]:
        selected_index = default_index
    else:
        continue
    # for j in range(default_index, len(thresholds)):
    #     if max_drop <= thresholds[j]*thres_mult:
    #         selected_index = j
    #         break


    if selected_index is None:
        continue  # No eligible contract found

    threshold = thresholds[selected_index]
    profit = profits[selected_index]
    loss = 100 - profit
    if daily_moves[i] <= threshold:
        result = 'WIN'
        PnL = profit
    else:
        result = 'LOSS'
        PnL = -loss
    total_pnl += PnL

    trade_log.append({
        "Date": df['Date'][i],
        "Threshold (%)": threshold,
        "Profit (%)": profit,
        "Loss (%)": loss,
        "PnL (%)": PnL,
        "Total PnL (%)": total_pnl,
        "Actual Move (%)": round(daily_moves[i], 2),
        "Result": result
    })

# Convert to DataFrame
trade_df = pd.DataFrame(trade_log)

# Total trades and win rate
total_trades = len(trade_df)
wins = (trade_df['Result'] == 'WIN').sum()
win_rate = wins / total_trades

# Expected value and reward
expected_value = (
    win_rate * trade_df['Profit (%)'].mean() -
    (1 - win_rate) * trade_df['Loss (%)'].mean()
)
reward = expected_value * total_trades

# Sharpe Ratio (annualized)
ann_factor = total_trades / 5
sharpe = trade_df['PnL (%)'].mean() / trade_df['PnL (%)'].std() * np.sqrt(ann_factor)

# Sortino Ratio (annualized)
downside_std = trade_df.loc[trade_df['PnL (%)'] < 0, 'PnL (%)'].std()
sortino = trade_df['PnL (%)'].mean() / downside_std * np.sqrt(ann_factor)

# Max Drawdown
# Linear cumulative PnL
cumulative = np.cumsum(trade_df['PnL (%)'])
print(list(cumulative))

# Rolling max of the cumulative PnL
rolling_max = np.maximum.accumulate(cumulative)

# Drawdown is absolute drop (not ratio!)
drawdown = cumulative - rolling_max  # will be X or negative

# Find the max drawdown (most negative drop)
max_drawdown = drawdown.min()  # e.g., -X means $X drop

# Optional: index (e.g., row number or trade number) of max drawdown
max_dd_index = drawdown.idxmin()

# CAGR from X years of compounding
initial_value = 1
final_value = cumulative.iloc[-1]
CAGR = (final_value / initial_value) ** (1 / 5) - 1  # 個人智慧財產,參數不提供 (placeholder)

fig, ax2 = plt.subplots(figsize=(10, 5))
# spy_prices = df["Change %"]
date = df.set_index("Date")["Change %"]
spy_returns = df["Change %"].cumsum()
print(spy_returns)
ax2.plot(df["Date"], df["Close"], color="tab:green")
ax2.set_ylabel('SPY Price', color='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:green')

ax1 = ax2.twinx()
ax1.plot(trade_df["Date"], trade_df["Total PnL (%)"], linestyle="--", label="Sell Call P&L", color="tab:red")
# axX.plot(combined.index, combined["Spike P&L"],      label="Spike P&L")
# axX.plot(combined.index, combined["Total P&L"],       linewidth=X, label="Total P&L")
ax1.set_xlabel("Date")
ax1.set_ylabel("Return")
ax1.tick_params(axis='y', labelcolor='tab:red')
ax1.grid(True)

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

# plt.figure(figsize=(X, X))
# spy_returns = df["Change %"].cumsum()
# dates = list(df["Date"])
# plt.plot(dates, spy_returns, linewidth=X, label="Cumulative Return")
# plt.plot(trade_df["Date"], trade_df["Total PnL (%)"], linewidth=X, label="Cumulative Return")
# plt.title("Cumulative PnL Over Time")
# plt.grid(True)
# plt.legend()
# plt.tight_layout()
# plt.show()

# Print results
print(f"Total Trades: {total_trades}")
print(f"Winrate: {win_rate:.2%}")
print(f"Expected Value: {expected_value:.2f}%")
print(f"Reward: {reward:.2f}%")
print(f"Sharpe Ratio: {sharpe:.2f}")
print(f"Sortino Ratio: {sortino:.2f}")
print(f"Max Drawdown: {max_drawdown:.2%} on {max_drawdown_date}")
print(f"CAGR: {CAGR:.2%}")
print(trade_df["Date"][277])
# Summary
summary_df = pd.DataFrame([{
    "Total Trades": total_trades,
    "Wins": wins,
    "Losses": losses,
    "Win Rate (%)": round(win_rate * 100, 2),
    "Average Profit (%)": round(trade_df['Profit (%)'].mean(), 2),
    "Average Loss (%)": round(trade_df['Loss (%)'].mean(), 2),
    "Expected Value per Trade (%)": round(expected_value, 2),
    "Total Reward": round(reward, 2)
}])

print(summary_df)

# Optional: Save trade log
# trade_df.to_csv("trade_log.csv", index=False)

# VIX trigger sell call sell put

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data
spy_df = pd.read_csv("SPY ETF Stock Price History 2021 to 2025.csv")
vix_df = pd.read_csv("CBOE Volatility Index Historical Data 2021 to 2025.csv")

# Preprocess SPY
spy_df.rename(columns={'Price': 'Close'}, inplace=True)
spy_df['Date'] = pd.to_datetime(spy_df['Date'])
spy_df = spy_df.sort_values('Date').reset_index(drop=True)
spy_df['Daily % Change'] = spy_df['Close'].pct_change() * 100
spy_df = spy_df.dropna().reset_index(drop=True)

# Preprocess VIX
vix_df['Date'] = pd.to_datetime(vix_df['Date'])
vix_df = vix_df.sort_values('Date').reset_index(drop=True)
vix_df.rename(columns={'Price': 'VIX_Close'}, inplace=True)
vix_df['VIX % Change'] = vix_df['VIX_Close'].pct_change() * 100

# Merge by date
merged = pd.merge(spy_df, vix_df, on='Date', how='inner')
daily_moves = merged['Daily % Change'].values
vix_values = merged['VIX_Close'].values
dates = merged['Date']
vix_moves = merged['VIX % Change'].values

# Strategy Parameters
lookback_days = LOOKBACK_DAYS
vix_threshold = VOL_CUTOFF
thres_mult = 1

# Thresholds and profits
thresholds = THRESHOLDS
profits = PROFITS
default_index_call = len(thresholds) // 2
default_index_put = 0

# Run strategy
trade_log = []
pnl = 0

for i in range(lookback_days, len(daily_moves)):
    max_drop = abs(min(daily_moves[i - lookback_days:i]))
    vix_today = vix_values[i]
    actual_move = daily_moves[i]
    vix_move = vix_moves[i]
    if vix_today > vix_threshold:
        direction = 'CALL'
        selected_index = None
        for j in range(default_index_call, len(thresholds)):
            if max_drop <= thresholds[j] * thres_mult:
                selected_index = j
                break
        if selected_index is None:
            continue
    else:
        direction = 'PUT'
        selected_index = default_index_put

    threshold = thresholds[selected_index]
    profit = profits[selected_index]
    loss = 100 - profit

    if direction == 'CALL':
        if actual_move <= threshold:
            result = 'WIN'
            pnl += profit
        else:
            result = 'LOSS'
            pnl -= loss
    else:
        if actual_move >= -threshold:
            result = 'WIN'
            pnl += profit
        else:
            result = 'LOSS'
            pnl -= loss

    trade_log.append({
        "Date": dates[i],
        "Direction": direction,
        "Threshold (%)": threshold,
        "Profit (%)": profit,
        "Loss (%)": loss,
        "Actual Move (%)": round(actual_move, 2),
        "VIX": vix_today,
        "Result": result,
        "pnl": pnl
    })

# Compile results
trade_df = pd.DataFrame(trade_log)
total_trades = len(trade_df)
wins = (trade_df['Result'] == 'WIN').sum()
losses = total_trades - wins
win_rate = wins / total_trades
expected_value = (win_rate * trade_df['Profit (%)'].mean()) - ((1 - win_rate) * trade_df['Loss (%)'].mean())
reward = expected_value * total_trades

# Summary
summary_df = pd.DataFrame([{
    "Total Trades": total_trades,
    "Wins": wins,
    "Losses": losses,
    "Win Rate (%)": round(win_rate * 100, 2),
    "Average Profit (%)": round(trade_df['Profit (%)'].mean(), 2),
    "Average Loss (%)": round(trade_df['Loss (%)'].mean(), 2),
    "Expected Value per Trade (%)": round(expected_value, 2),
    "Total Reward": round(reward, 2)
}])

print(summary_df)

# Optionally save trade details
# trade_df.to_csv("vix_call_put_strategy_with_fixed_put.csv", index=False)

# Plot the trade returns
plt.figure(figsize=(12, 6))
plt.plot(trade_df["Date"], trade_df["pnl"], label="Trade P&L", marker='o')
plt.title("Non-Spike Strategy Trade Performance")
plt.xlabel("Date")
plt.ylabel("Return")
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
